# Sold Transactions Analysis
## Step 1: Imported Libraries

In [22]:
import pandas as pd
import glob
import os

## Step 2: Loaded Sold CSV Files

In [23]:
sold_files = glob.glob("../data/CRMLSSold*.csv")
print(len(sold_files))
print(sold_files)

27
['../data\\CRMLSSold202401.csv', '../data\\CRMLSSold202402.csv', '../data\\CRMLSSold202403.csv', '../data\\CRMLSSold202404.csv', '../data\\CRMLSSold202405.csv', '../data\\CRMLSSold202406.csv', '../data\\CRMLSSold202407.csv', '../data\\CRMLSSold202408.csv', '../data\\CRMLSSold202409.csv', '../data\\CRMLSSold202410.csv', '../data\\CRMLSSold202411.csv', '../data\\CRMLSSold202412.csv', '../data\\CRMLSSold202501.csv', '../data\\CRMLSSold202502.csv', '../data\\CRMLSSold202503.csv', '../data\\CRMLSSold202504.csv', '../data\\CRMLSSold202505.csv', '../data\\CRMLSSold202506.csv', '../data\\CRMLSSold202507.csv', '../data\\CRMLSSold202508.csv', '../data\\CRMLSSold202509.csv', '../data\\CRMLSSold202510.csv', '../data\\CRMLSSold202511.csv', '../data\\CRMLSSold202512.csv', '../data\\CRMLSSold202601.csv', '../data\\CRMLSSold202602.csv', '../data\\CRMLSSold202603.csv']


## Step 3: Explored a Single File
Inspected one file before combining - check shape, columns, data types, and sample rows.

In [24]:
df_sample = pd.read_csv(sold_files[0])
print (df_sample.shape)
print(df_sample.columns.tolist())
print(df_sample.dtypes)
print(df_sample.head(3))

(17976, 80)
['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount', 'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City', 'TaxYear', 'BuildingAreaTotal', 'BedroomsTotal', 'ContractStatusChangeDate', 'Elementa

## Step 4: Combined All 27 Sold Files
Stacked all monthly CSVs (Jan 2024 - Mar 2026) into one DataFrame.

In [25]:
dfs = [pd.read_csv(f, low_memory= False, encoding = 'latin-1') for f in sold_files]
df_sold = pd.concat(dfs, ignore_index=True)
print(df_sold.shape)
print(df_sold.columns.tolist())

(591475, 84)
['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount', 'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City', 'TaxYear', 'BuildingAreaTotal', 'BedroomsTotal', 'ContractStatusChangeDate', 'Element

## Step 5: Explored the Combined Dataset
Checked null counts and unique values in key categorical fields.

In [26]:
print(df_sold.isnull().sum())
df_sold['PropertyType'].unique()

BuyerAgentAOR                   71536
ListAgentAOR                    68048
Flooring                       243802
ViewYN                          59249
WaterfrontYN                   591153
                                ...  
OriginatingSystemSubName       531106
BuyerAgencyCompensationType    523728
BuyerAgencyCompensation        523695
latfilled                      495682
lonfilled                      495682
Length: 84, dtype: int64


<StringArray>
[        'Residential',     'CommercialLease',                'Land',
    'ResidentialLease',  'ManufacturedInPark',   'ResidentialIncome',
      'CommercialSale', 'BusinessOpportunity']
Length: 8, dtype: str

In [27]:
critical_cols = ['ClosePrice', 'ListPrice', 'OriginalListPrice', 'LivingArea', 'DaysOnMarket', 'PropertyType', 'CloseDate', 'City', 'PostalCode', 'BedroomsTotal', 'BathroomsTotalInteger']
print(df_sold[critical_cols].isnull().sum())

ClosePrice                   7
ListPrice                  910
OriginalListPrice         1715
LivingArea               41756
DaysOnMarket                 0
PropertyType                 0
CloseDate                    0
City                       440
PostalCode                 165
BedroomsTotal            39388
BathroomsTotalInteger    27418
dtype: int64


## Step 6: Cleaned the Data
- Filtered to Residential only
- Dropped rows missing ClosePrice or LivingArea
- Converted date columns to datetime
- Dropped columns that were mostly null or irrelevant

In [28]:
df_sold = df_sold[df_sold['PropertyType'] == 'Residential']
df_sold = df_sold.dropna(subset=['ClosePrice', 'LivingArea'])
print(df_sold.shape)

(397232, 84)


In [29]:
df_sold['CloseDate'] = pd.to_datetime(df_sold['CloseDate'])
df_sold['ListingContractDate'] = pd.to_datetime(df_sold['ListingContractDate'])
df_sold['ContractStatusChangeDate'] = pd.to_datetime(df_sold['ContractStatusChangeDate'])
df_sold['PurchaseContractDate'] = pd.to_datetime(df_sold['PurchaseContractDate'])
print(df_sold[['CloseDate', 'ListingContractDate', 'ContractStatusChangeDate', 'PurchaseContractDate']].dtypes)

CloseDate                   datetime64[us]
ListingContractDate         datetime64[us]
ContractStatusChangeDate    datetime64[us]
PurchaseContractDate        datetime64[us]
dtype: object


In [30]:
cols_to_drop = [
    'WaterfrontYN', 'latfilled', 'lonfilled',
    'BuyerAgencyCompensationType', 'BuyerAgencyCompensation',
    'OriginatingSystemSubName', 'BusinessType', 'MiddleOrJuniorSchoolDistrict', 'TaxYear', 'FireplacesTotal', 'TaxAnnualAmount','AboveGradeFinishedArea'
]
df_sold = df_sold.drop(columns=cols_to_drop)
print(df_sold.shape)

(397232, 72)


In [31]:
df_sold.isnull().sum().sort_values(ascending=False)

ElementarySchoolDistrict    397232
CoveredSpaces               397232
BelowGradeFinishedArea      394942
BasementYN                  389468
LotSizeDimensions           378017
                             ...  
PropertyType                     0
DaysOnMarket                     0
StateOrProvince                  0
ListingId                        0
BedroomsTotal                    0
Length: 72, dtype: int64

In [32]:
df_sold.select_dtypes(include='object').columns.tolist()

C:\Users\Sanjana\AppData\Local\Temp\ipykernel_225288\3411693468.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_sold.select_dtypes(include='object').columns.tolist()


['BuyerAgentAOR',
 'ListAgentAOR',
 'Flooring',
 'ViewYN',
 'BasementYN',
 'PoolPrivateYN',
 'ListAgentEmail',
 'ListAgentFirstName',
 'ListAgentLastName',
 'UnparsedAddress',
 'PropertyType',
 'ListOfficeName',
 'BuyerOfficeName',
 'CoListOfficeName',
 'ListAgentFullName',
 'CoListAgentFirstName',
 'CoListAgentLastName',
 'BuyerAgentMlsId',
 'BuyerAgentFirstName',
 'BuyerAgentLastName',
 'AssociationFeeFrequency',
 'MLSAreaMajor',
 'CountyOrParish',
 'MlsStatus',
 'ElementarySchool',
 'AttachedGarageYN',
 'BuilderName',
 'PropertySubType',
 'SubdivisionName',
 'BuyerOfficeAOR',
 'ListingId',
 'City',
 'CoBuyerAgentFirstName',
 'StateOrProvince',
 'MiddleOrJuniorSchool',
 'FireplaceYN',
 'HighSchool',
 'Levels',
 'LotSizeDimensions',
 'NewConstructionYN',
 'HighSchoolDistrict',
 'PostalCode',
 'OriginatingSystemName']

In [33]:
print(df_sold.shape)

(397232, 72)


## Step 7: Feature Engineering
Created new calculated columns: PriceRatio, PricePerSqFt, CloseYear, CloseMonth, YrMo.